# Classificadores de imagens pré-treinados: AlexNet, GoogLeNet, VGG16 e Vision Transformer

**Disciplina:** Processamento Digital de Sinais e Imagens (PDSI)

## Contexto

Este notebook usa **quatro modelos pré-treinados no ImageNet** (1000 classes) para classificar a mesma imagem. São arquiteturas que vimos no notebook *M3 - CNNs architectures* (`print(dir(TModels))`):

| Modelo | Ano | Ideia principal |
|---|---|---|
| **AlexNet** | 2012 | Primeira CNN profunda a vencer o ImageNet: ReLU, *dropout* e treino em GPU |
| **GoogLeNet** | 2014 | Módulos *Inception*: convoluções de vários tamanhos em paralelo |
| **VGG16** | 2014 | Rede simples e profunda, só com filtros 3×3 empilhados |
| **Vision Transformer (ViT)** | 2020 | Não é CNN: divide a imagem em *patches* 16×16 e usa *self-attention* |

## Como usar
1. Faça o upload de uma foto de um elemento do mundo real (cachorro, gato, carro, xícara...) na pasta `imagens` e ajuste o `CAMINHO_IMAGEM` na Parte 1.
2. Execute todas as células e compare as previsões dos quatro modelos.
3. Consulte a documentação indicada em cada modelo e no final do notebook.

> 💡 Os modelos VGG16 e ViT são grandes (cerca de 530 MB e 330 MB). No Colab, o download dos pesos acontece só na primeira execução.


In [ ]:
# Download do OpenCV
!pip install opencv-python

## Parte 1 – Bibliotecas e imagem de entrada

In [ ]:
import os
# Importa o PyTorch e o módulo de modelos prontos do torchvision
import torch
import torchvision.models as TModels
# Bibliotecas de apoio: NumPy, OpenCV, Matplotlib e PIL
import numpy as np
import cv2
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from PIL import Image
import time

# Usa a GPU se estiver disponível; caso contrário, roda na CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", device)

In [ ]:
# Download do repositório
if os.path.isdir("pdsi"):
  print("Diretório já existe!")
else:
  !git clone https://github.com/NaldoHoracio/pdsi

In [ ]:
CAMINHO_IMAGEM = "./pdsi/imagens/Cat.png"

# Lê a imagem (OpenCV usa BGR) e converte para RGB, ordem usada no treino dos modelos
img_bgr = cv2.imread(CAMINHO_IMAGEM)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

# Exibe a imagem com suas dimensões (altura, largura, canais)
plt.imshow(img_rgb)
plt.title(f"Imagem original {img_rgb.shape}")
plt.axis("off")
plt.show()

## Parte 2 – Funções de apoio

Todos os modelos seguem os mesmos passos: **pré-processar → inferência → softmax → top-5**. Para não repetir código, reunimos esses passos em funções.

In [ ]:
# Conta o total de parâmetros (pesos e vieses) de um modelo
def contar_parametros(modelo):
    return sum(p.numel() for p in modelo.parameters())

# Pré-processa a imagem, roda o modelo e devolve as 5 classes mais prováveis e o tempo de inferência
def classificar(modelo, pesos, img_rgb, k=5):
    preprocess = pesos.transforms()# pré-processamento usado no treino desses pesos
    lote = preprocess(Image.fromarray(img_rgb)).unsqueeze(0).to(device) # tensor (1, 3, H, W)
    with torch.no_grad():# inferência sem cálculo de gradientes
        inicio = time.perf_counter()
        logits = modelo(lote)
        tempo = time.perf_counter() - inicio
    probs = torch.softmax(logits, dim=1)[0]# logits -> probabilidades
    top_p, top_i = torch.topk(probs, k)# k classes mais prováveis
    categorias = pesos.meta["categories"]
    return [categorias[i] for i in top_i.tolist()], top_p.cpu().numpy(), tempo

# Gráfico de barras horizontais com as 5 classes mais prováveis
def plotar_top5(nomes, valores, titulo, cor="seagreen"):
    nomes, valores = nomes[::-1], valores[::-1]# a classe mais provável fica no topo
    plt.figure(figsize=(6, 3))
    barras = plt.barh(nomes, valores, color=cor)
    plt.bar_label(barras, labels=[f"{v:.1%}" for v in valores], padding=3)
    plt.xlim(0, valores.max() * 1.2)
    plt.gca().xaxis.set_major_formatter(PercentFormatter(xmax=1))
    plt.xlabel("Probabilidade", fontsize=8); plt.ylabel("Classes", fontsize=8)
    plt.title(titulo, fontsize=10); plt.tight_layout(); plt.show()

resultados = {}# guarda os resultados de cada modelo para a comparação final

## Parte 3 – AlexNet

**AlexNet** (Krizhevsky, Sutskever e Hinton, 2012) venceu a competição ImageNet com grande vantagem e marcou o início da era do *deep learning* em visão computacional. Tem 5 camadas convolucionais e 3 totalmente conectadas, usa ReLU, *dropout* e foi treinada em GPUs.

📄 Documentação: https://pytorch.org/vision/stable/models/generated/torchvision.models.alexnet.html

In [ ]:
# Seleciona os melhores pesos disponíveis, treinados no ImageNet-1K
pesos_alexnet = TModels.AlexNet_Weights.DEFAULT
# Cria o modelo com os pesos pré-treinados, em modo de avaliação, no dispositivo escolhido
modelo_alexnet = TModels.alexnet(weights=pesos_alexnet).eval().to(device)

# Mostra o número de parâmetros e o pré-processamento esperado pelo modelo
print(f"Parâmetros da AlexNet: {contar_parametros(modelo_alexnet):,}")
print(pesos_alexnet.transforms())

In [ ]:
# Classifica a imagem e guarda o resultado para a comparação final
nomes, valores, tempo = classificar(modelo_alexnet, pesos_alexnet, img_rgb)
resultados["AlexNet"] = {"classes": nomes, "probs": valores, "tempo": tempo, "params": contar_parametros(modelo_alexnet)}

# Mostra as 5 previsões e o tempo de inferência
print(f"Top-5 previsões (AlexNet) - inferência em {tempo * 1000:.1f} ms:")
for c, p in zip(nomes, valores):
    print(f"  {c:25s} {p:6.1%}")
plotar_top5(nomes, valores, "Prováveis classes da imagem usando AlexNet (top 5)", cor="tab:blue")

## Parte 4 – GoogLeNet

**GoogLeNet** (Szegedy et al., 2014), também chamada de Inception v1, venceu o ImageNet de 2014. Seu bloco principal, o **módulo Inception**, aplica convoluções 1×1, 3×3 e 5×5 e um *max pooling* **em paralelo** e concatena os resultados. Com 22 camadas, tem bem menos parâmetros que a AlexNet, graças às convoluções 1×1 que reduzem o número de canais.

📄 Documentação: https://pytorch.org/vision/stable/models/generated/torchvision.models.googlenet.html

In [ ]:
# Seleciona os melhores pesos disponíveis, treinados no ImageNet-1K
pesos_googlenet = TModels.GoogLeNet_Weights.DEFAULT
# Cria o modelo com os pesos pré-treinados, em modo de avaliação, no dispositivo escolhido
modelo_googlenet = TModels.googlenet(weights=pesos_googlenet).eval().to(device)

# Mostra o número de parâmetros e o pré-processamento esperado pelo modelo
print(f"Parâmetros da GoogLeNet: {contar_parametros(modelo_googlenet):,}")
print(pesos_googlenet.transforms())

In [ ]:
# Classifica a imagem e guarda o resultado para a comparação final
nomes, valores, tempo = classificar(modelo_googlenet, pesos_googlenet, img_rgb)
resultados["GoogLeNet"] = {"classes": nomes, "probs": valores, "tempo": tempo, "params": contar_parametros(modelo_googlenet)}

# Mostra as 5 previsões e o tempo de inferência
print(f"Top-5 previsões (GoogLeNet) - inferência em {tempo * 1000:.1f} ms:")
for c, p in zip(nomes, valores):
    print(f"  {c:25s} {p:6.1%}")
plotar_top5(nomes, valores, "Prováveis classes da imagem usando GoogLeNet (top 5)", cor="tab:orange")

## Parte 5 – VGG16

**VGG16** (Simonyan e Zisserman, 2014) mostrou que profundidade importa: usa **apenas filtros 3×3** empilhados (13 camadas convolucionais + 3 totalmente conectadas). A arquitetura é simples e regular, mas pesada: a maior parte dos parâmetros está nas camadas totalmente conectadas.

📄 Documentação: https://pytorch.org/vision/stable/models/generated/torchvision.models.vgg16.html

In [ ]:
# Seleciona os melhores pesos disponíveis, treinados no ImageNet-1K
pesos_vgg16 = TModels.VGG16_Weights.DEFAULT
# Cria o modelo com os pesos pré-treinados, em modo de avaliação, no dispositivo escolhido
modelo_vgg16 = TModels.vgg16(weights=pesos_vgg16).eval().to(device)

# Mostra o número de parâmetros e o pré-processamento esperado pelo modelo
print(f"Parâmetros da VGG16: {contar_parametros(modelo_vgg16):,}")
print(pesos_vgg16.transforms())

In [ ]:
# Classifica a imagem e guarda o resultado para a comparação final
nomes, valores, tempo = classificar(modelo_vgg16, pesos_vgg16, img_rgb)
resultados["VGG16"] = {"classes": nomes, "probs": valores, "tempo": tempo, "params": contar_parametros(modelo_vgg16)}

# Mostra as 5 previsões e o tempo de inferência
print(f"Top-5 previsões (VGG16) - inferência em {tempo * 1000:.1f} ms:")
for c, p in zip(nomes, valores):
    print(f"  {c:25s} {p:6.1%}")
plotar_top5(nomes, valores, "Prováveis classes da imagem usando VGG16 (top 5)", cor="tab:purple")

## Parte 6 – ViT-B/16

**Vision Transformer** (Dosovitskiy et al., 2020) **não é uma CNN**. A imagem é dividida em *patches* de 16×16 pixels, cada *patch* vira um vetor (como uma "palavra") e uma arquitetura **Transformer**, criada para textos, usa *self-attention* para relacionar todos os *patches* entre si. Usamos a versão *Base* com *patches* de 16 (`vit_b_16`).

📄 Documentação: https://pytorch.org/vision/stable/models/generated/torchvision.models.vit_b_16.html

In [ ]:
# Seleciona os melhores pesos disponíveis, treinados no ImageNet-1K
pesos_vit = TModels.ViT_B_16_Weights.DEFAULT
# Cria o modelo com os pesos pré-treinados, em modo de avaliação, no dispositivo escolhido
modelo_vit = TModels.vit_b_16(weights=pesos_vit).eval().to(device)

# Mostra o número de parâmetros e o pré-processamento esperado pelo modelo
print(f"Parâmetros da ViT-B/16: {contar_parametros(modelo_vit):,}")
print(pesos_vit.transforms())

In [ ]:
# Classifica a imagem e guarda o resultado para a comparação final
nomes, valores, tempo = classificar(modelo_vit, pesos_vit, img_rgb)
resultados["ViT-B/16"] = {"classes": nomes, "probs": valores, "tempo": tempo, "params": contar_parametros(modelo_vit)}

# Mostra as 5 previsões e o tempo de inferência
print(f"Top-5 previsões (ViT-B/16) - inferência em {tempo * 1000:.1f} ms:")
for c, p in zip(nomes, valores):
    print(f"  {c:25s} {p:6.1%}")
plotar_top5(nomes, valores, "Prováveis classes da imagem usando ViT-B/16 (top 5)", cor="tab:red")

## Parte 7 – Comparação entre os modelos

In [ ]:
# Monta uma tabela com a classe mais provável, a confiança, o tamanho e o tempo de cada modelo
print(f"{'Modelo':10s} {'Top-1':25s} {'Confiança':>10s} {'Parâmetros':>14s} {'Tempo (ms)':>11s}")
for nome, r in resultados.items():
    print(f"{nome:10s} {r['classes'][0]:25s} {r['probs'][0]:10.1%} {r['params']:14,} {r['tempo'] * 1000:11.1f}")

In [ ]:
# Gráficos lado a lado: número de parâmetros e confiança na classe top-1 de cada modelo
nomes_modelos = list(resultados.keys())
cores = ["tab:blue", "tab:orange", "tab:purple", "tab:red"]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))

barras = axes[0].bar(nomes_modelos, [r["params"] / 1e6 for r in resultados.values()], color=cores)
axes[0].bar_label(barras, fmt="%.1f M", padding=3)
axes[0].set_ylabel("Milhões de parâmetros"); axes[0].set_title("Tamanho do modelo")
axes[0].set_ylim(0, max(r["params"] for r in resultados.values()) / 1e6 * 1.2)

barras = axes[1].bar(nomes_modelos, [r["probs"][0] for r in resultados.values()], color=cores)
axes[1].bar_label(barras, labels=[f"{r['classes'][0]}\n{r['probs'][0]:.1%}" for r in resultados.values()], padding=3, fontsize=8)
axes[1].yaxis.set_major_formatter(PercentFormatter(xmax=1)); axes[1].set_ylim(0, 1.25)
axes[1].set_title("Classe top-1 e confiança")
plt.tight_layout(); plt.show()

### Perguntas para discussão

1. Os quatro modelos concordaram sobre a classe da imagem? Algum ficou mais "confiante" que os outros?
2. A VGG16 tem muito mais parâmetros que a GoogLeNet. Isso garantiu uma previsão melhor? O que isso diz sobre a importância da **arquitetura** em relação ao **tamanho** do modelo?
3. Compare o pré-processamento (`transforms()`) dos modelos. Todos usam o mesmo tamanho de entrada e a mesma normalização?
4. Qual é a diferença fundamental entre o ViT e as três CNNs na forma de processar a imagem?
5. Teste uma imagem difícil (objeto parcialmente escondido, ângulo incomum ou pouca luz). Qual modelo se saiu melhor?

## Referências para consulta

- **Lista de todos os modelos pré-treinados do torchvision** e como usar os pesos: https://pytorch.org/vision/stable/models.html
- KRIZHEVSKY, A.; SUTSKEVER, I.; HINTON, G. E. *ImageNet Classification with Deep Convolutional Neural Networks*. In: Advances in Neural Information Processing Systems (NeurIPS), 2012. (AlexNet)
- SZEGEDY, C. et al. *Going Deeper with Convolutions*. In: IEEE Conference on Computer Vision and Pattern Recognition (CVPR), 2015. Disponível em: https://arxiv.org/abs/1409.4842 (GoogLeNet)
- SIMONYAN, K.; ZISSERMAN, A. *Very Deep Convolutional Networks for Large-Scale Image Recognition*. In: International Conference on Learning Representations (ICLR), 2015. Disponível em: https://arxiv.org/abs/1409.1556 (VGG)
- DOSOVITSKIY, A. et al. *An Image is Worth 16x16 Words: Transformers for Image Recognition at Scale*. In: International Conference on Learning Representations (ICLR), 2021. Disponível em: https://arxiv.org/abs/2010.11929 (Vision Transformer)
